In [26]:
SEED = 42

In [13]:
from pathlib import Path
from helpers.data.ticker_loader import load_tickers

TICKERS_FILE = Path("tickers.txt")
if not TICKERS_FILE.exists():
    TICKERS_FILE = Path("NN_Trading_project/tickers.txt")

# None        → all tickers
# ["AAPL", …] → explicit list
# 42          → random sample of 42 (seeded by SEED)
TICKER_SUBSET = None

TICKERS = load_tickers(TICKERS_FILE, subset=TICKER_SUBSET, seed=SEED)
print(f"Using {len(TICKERS)} tickers")

Using 61 tickers


In [14]:
import pandas as pd
import datetime

INTERVAL   = "1d"
START_DATE = pd.Timestamp("2025-01-01")
END_DATE   = pd.Timestamp(datetime.date.today())

# Train: Date <= TRAIN_END_DATE
# Val:   (TRAIN_END_DATE, VAL_END_DATE]
# Test:  Date > VAL_END_DATE
TRAIN_END_DATE = pd.Timestamp("2025-11-30")
VAL_END_DATE   = pd.Timestamp("2026-01-20")

REBUILD_FEATURE_CACHE = True

In [15]:
import datetime
from pathlib import Path

RUN_START_TIME = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_OUTPUT_DIR = Path.cwd() / "run_output" / RUN_START_TIME
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run output directory: {RUN_OUTPUT_DIR}")


Run output directory: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/run_output/20260414_162450


In [16]:
# Trading / Labeling
HORIZON_BARS     = 5
PROFIT_THRESHOLD = 2 / 100
STOP_LOSS        = -1 / 100
WINDOW           = 20

# Training
MAX_EPOCHS    = 50
PATIENCE      = 10
BUY_THRESHOLD = 0.5

# Optimizer / Model
BATCH_SIZE   = 64

# DataLoader
SPLIT_FRAC  = 0.85
NUM_WORKERS = 16

In [17]:
from helpers.data.date_config_manager import check_and_refresh_date_config

check_and_refresh_date_config(
    current_config={
        "INTERVAL":           str(INTERVAL),
        "START_DATE":         str(START_DATE.date()),
        "END_DATE":           str(END_DATE.date()),
        "TRAIN_END_DATE":     str(TRAIN_END_DATE.date()),
        "VAL_END_DATE":       str(VAL_END_DATE.date()),
        "TICKER_SUBSET":      str(TICKER_SUBSET),
    },
    config_path=Path.cwd() / "date_config.txt",
    stocks_dir=Path.cwd() / "dataset" / "stocks",
)

Date config changed — clearing cached CSVs for a fresh download.
  Previous config:
    INTERVAL: 1d
    START_DATE: 2025-01-01
    END_DATE: 2026-04-14
    TRAIN_END_DATE: 2025-11-30
    VAL_END_DATE: 2026-01-20
    TICKER_SUBSET: 10 <-- CHANGED
  Deleted: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset/stocks
  Updated: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/date_config.txt


True

In [18]:
from helpers.data.data_downloader import download_tickers

data_root  = Path.cwd() / "dataset"
stocks_dir = data_root / "stocks"

_summary = download_tickers(
    tickers=TICKERS, start=START_DATE, end=END_DATE,
    interval=INTERVAL, out_dir=stocks_dir,
)

 OK   :: AGCO rows=319 | downloaded 1/61
 OK   :: ALLE rows=319 | downloaded 2/61
 OK   :: ALNY rows=319 | downloaded 3/61
 OK   :: AVB rows=319 | downloaded 4/61
 OK   :: BR rows=319 | downloaded 5/61
 OK   :: BURL rows=319 | downloaded 6/61
 OK   :: BWXT rows=319 | downloaded 7/61
 OK   :: CALM rows=319 | downloaded 8/61
 OK   :: CBOE rows=319 | downloaded 9/61
 OK   :: CMI rows=319 | downloaded 10/61
 OK   :: CYBR rows=280 | downloaded 11/61
 OK   :: DGX rows=319 | downloaded 12/61
 OK   :: DOV rows=319 | downloaded 13/61
 OK   :: EWBC rows=319 | downloaded 14/61
 OK   :: EXR rows=319 | downloaded 15/61
 OK   :: FAF rows=319 | downloaded 16/61
 OK   :: FVRR rows=319 | downloaded 17/61
 OK   :: GGG rows=319 | downloaded 18/61
 OK   :: HIW rows=319 | downloaded 19/61
 OK   :: HOPE rows=319 | downloaded 20/61
 OK   :: HXL rows=319 | downloaded 21/61
 OK   :: IAC rows=319 | downloaded 22/61
 OK   :: IT rows=319 | downloaded 23/61
 OK   :: ITW rows=319 | downloaded 24/61
 OK   :: JBHT ro

# Building Features + Labels

In [19]:
import pickle
import shutil
import time

import torch
from torch.utils.data import DataLoader

from helpers.feature.feature_builder import precompute_and_cache, FEATURE_COLS
from helpers.data.dataset import StockDatasetSafe, is_cache_valid

root       = Path.cwd() / "dataset"
stocks_dir = root / "stocks"
assert stocks_dir.exists(), f"Missing: {stocks_dir}"

files       = sorted(stocks_dir.glob("*.csv"))
cache_dir   = Path.cwd() / f".feature_cache_forward_return_w{WINDOW}"
cache_dir.mkdir(parents=True, exist_ok=True)
scaler_path = cache_dir / "scaler.pkl"
index_path  = cache_dir / "index.pkl"

if REBUILD_FEATURE_CACHE:
    for p in [pp for pp in Path.cwd().glob(".feature*") if pp.exists()]:
        shutil.rmtree(p, ignore_errors=True) if p.is_dir() else p.unlink(missing_ok=True)
        print(f"[Cache] Removed: {p}")
    time.sleep(1)

cache_ready = is_cache_valid(scaler_path, index_path)
if not cache_ready and cache_dir.exists():
    print("[Cache] Stale / incomplete — wiping cache dir.")
    shutil.rmtree(cache_dir)
cache_dir.mkdir(parents=True, exist_ok=True)

if REBUILD_FEATURE_CACHE or not cache_ready:
    scaler, index = precompute_and_cache(
        files=files, window=WINDOW, cache_dir=cache_dir,
        scaler_path=scaler_path, index_path=index_path,
        horizon_bars=HORIZON_BARS, train_end_date=TRAIN_END_DATE,
        val_end_date=VAL_END_DATE, profit_threshold=PROFIT_THRESHOLD,
        stop_loss=STOP_LOSS,
    )
else:
    print("[Cache] Using existing feature cache.")
    with open(scaler_path, "rb") as f: scaler = pickle.load(f)
    with open(index_path,  "rb") as f: index  = pickle.load(f)

train_ds = StockDatasetSafe(index, scaler, "train")
val_ds   = StockDatasetSafe(index, scaler, "val")
test_ds  = StockDatasetSafe(index, scaler, "test")

_pin    = torch.cuda.is_available()
assert _pin, "GPU required but torch.cuda.is_available() is False."
_kwargs = dict(
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    pin_memory=_pin, persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
)
train_loader = DataLoader(train_ds, shuffle=True,  **_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_kwargs)

xb, yb = next(iter(train_loader))
print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}")
print(f"X batch: {xb.shape} {xb.dtype}  y batch: {yb.shape} {yb.dtype}")
print(f"Features: {len(FEATURE_COLS)} base × {WINDOW} lags = {len(FEATURE_COLS) * WINDOW}")

[Cache] Removed: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/.feature_cache_forward_return_w20
[Cache] Precomputing features (leakage-safe splits)...
[Cache] (1/61) AGCO.csv
[Cache] (2/61) ALLE.csv
[Cache] (3/61) ALNY.csv
[Cache] (4/61) AVB.csv
[Cache] (5/61) BR.csv
[Cache] (6/61) BURL.csv
[Cache] (7/61) BWXT.csv
[Cache] (8/61) CALM.csv
[Cache] (9/61) CBOE.csv
[Cache] (10/61) CMI.csv
[Cache] (11/61) CYBR.csv
[Cache] (12/61) DGX.csv
[Cache] (13/61) DOV.csv
[Cache] (14/61) EWBC.csv
[Cache] (15/61) EXR.csv
[Cache] (16/61) FAF.csv
[Cache] (17/61) FVRR.csv
[Cache] (18/61) GGG.csv
[Cache] (19/61) HIW.csv
[Cache] (20/61) HOPE.csv
[Cache] (21/61) HXL.csv
[Cache] (22/61) IAC.csv
[Cache] (23/61) IT.csv
[Cache] (24/61) ITW.csv
[Cache] (25/61) JBHT.csv
[Cache] (26/61) KLAC.csv
[Cache] (27/61) KMT.csv
[Cache] (28/61) LAZ.csv
[Cache] (29/61) LOGI.csv
[Cache] (30/61) LPLA.csv
[Cache] (31/61) MAA.csv
[Cache] (32/61) MCK.csv
[Cache] (33/61) MCO.csv
[Cache] (34/61) MLI.csv
[Cache] (35/61) M

# XGBoost

In [20]:
import numpy as np, torch, xgboost as xgb, joblib
from sklearn.metrics import log_loss
from helpers.evaluation import buy_metrics, predict_probs_booster

# ── Config ───────────────────────────────────────────────────────────────────
NUM_BOOST_ROUND      = 10000
EARLY_STOPPING_ROUNDS = PATIENCE

def loader_to_numpy(loader):
    Xs, ys = zip(*[(xb.numpy(), yb.numpy()) for xb, yb in loader])
    return np.concatenate(Xs), np.concatenate(ys)

X_train, y_train = loader_to_numpy(train_loader)
X_val,   y_val   = loader_to_numpy(val_loader)
X_test,  y_test  = loader_to_numpy(test_loader)

num_pos, num_neg = float((y_train == 1).sum()), float((y_train == 0).sum())
scale_pos_weight = num_neg / max(1.0, num_pos)
print(f"Train {X_train.shape}  pos={int(num_pos)} neg={int(num_neg)}")
print(f"Val   {X_val.shape}    pos={int((y_val==1).sum())} neg={int((y_val==0).sum())}")
print(f"Test  {X_test.shape}   pos={int((y_test==1).sum())} neg={int((y_test==0).sum())}")
print(f"scale_pos_weight = {scale_pos_weight:.4f}")

GPU_REQUIRED_MSG        = "GPU required but torch.cuda.is_available() is False."
GPU_XGB_UNAVAILABLE_MSG = "GPU required but XGBoost CUDA training is unavailable. Install GPU-enabled XGBoost/CUDA or run on a GPU machine."
GPU_XGB_PARAMS          = {"tree_method": "hist", "device": "cuda"}

def assert_cuda_available():
    assert torch.cuda.is_available(), GPU_REQUIRED_MSG

def assert_xgb_cuda_training_available(X, y):
    dtmp = xgb.DMatrix(X[:min(2048, len(X))], label=y[:min(2048, len(y))])
    try:
        xgb.train({"objective": "binary:logistic", "eval_metric": "logloss", "max_depth": 1, "eta": 0.3, **GPU_XGB_PARAMS}, dtmp, num_boost_round=1, verbose_eval=False)
    except Exception as e:
        raise AssertionError(GPU_XGB_UNAVAILABLE_MSG) from e

assert_cuda_available()
assert_xgb_cuda_training_available(X_train, y_train)

params = {
    "max_depth": 8, "eta": 0.0033968017011338967, "subsample": 0.9731797467870894,
    "colsample_bytree": 0.9095940838020883, "min_child_weight": 1, "gamma": 0.2774238622335642,
    "alpha": 4.2050762518663145, "lambda": 2.5755643313408796,
    "scale_pos_weight": scale_pos_weight, "objective": "binary:logistic", "eval_metric": "logloss",
    **GPU_XGB_PARAMS,
}

dtrain, dval = xgb.DMatrix(X_train, label=y_train), xgb.DMatrix(X_val, label=y_val)
print(f"Training: max_rounds={NUM_BOOST_ROUND}, early_stop={EARLY_STOPPING_ROUNDS}")
booster = xgb.train(params=params, dtrain=dtrain, num_boost_round=NUM_BOOST_ROUND,
                    evals=[(dtrain, "train"), (dval, "val")],
                    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                    verbose_eval=100)

best_ntree = int(booster.best_iteration + 1) if booster.best_iteration is not None else NUM_BOOST_ROUND
print(f"\nBest iteration: {best_ntree}")

probs_train = predict_probs_booster(booster, X_train, best_ntree)
probs_val   = predict_probs_booster(booster, X_val,   best_ntree)
probs_test  = predict_probs_booster(booster, X_test,  best_ntree)
tr = buy_metrics(y_train, probs_train, BUY_THRESHOLD)
vl = buy_metrics(y_val,   probs_val,   BUY_THRESHOLD)
te = buy_metrics(y_test,  probs_test,  BUY_THRESHOLD)

print(f"Train  logloss={log_loss(y_train,probs_train):.6f}  acc={tr['acc']:.2f}%  P(success|BUY)={tr['buy_success']:.2f}%")
print(f"Val    logloss={log_loss(y_val,  probs_val  ):.6f}  acc={vl['acc']:.2f}%  P(success|BUY)={vl['buy_success']:.2f}%")
print(f"Test   logloss={log_loss(y_test, probs_test ):.6f}  acc={te['acc']:.2f}%  P(success|BUY)={te['buy_success']:.2f}%")

MODEL_PATH = RUN_OUTPUT_DIR / "best_model_xgb.pkl"
joblib.dump({"booster": booster, "best_ntree": best_ntree}, MODEL_PATH)
print(f"Saved → {MODEL_PATH}")


Train (7991, 880)  pos=2439 neg=5552
Val   (1769, 880)    pos=602 neg=1167
Test  (3133, 880)   pos=925 neg=2208
scale_pos_weight = 2.2763
Training: max_rounds=10000, early_stop=10
[0]	train-logloss:0.69236	val-logloss:0.69303
[100]	train-logloss:0.62621	val-logloss:0.68439
[200]	train-logloss:0.57329	val-logloss:0.67694
[300]	train-logloss:0.52839	val-logloss:0.67161
[400]	train-logloss:0.48792	val-logloss:0.66709
[500]	train-logloss:0.45101	val-logloss:0.66371
[600]	train-logloss:0.42024	val-logloss:0.66128
[700]	train-logloss:0.39312	val-logloss:0.65898
[800]	train-logloss:0.36900	val-logloss:0.65766
[813]	train-logloss:0.36621	val-logloss:0.65765

Best iteration: 804
Train  logloss=0.368391  acc=99.79%  P(success|BUY)=99.39%
Val    logloss=0.657575  acc=61.79%  P(success|BUY)=31.50%
Test   logloss=0.654041  acc=63.17%  P(success|BUY)=28.76%
Saved → /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/run_output/20260414_162450/best_model_xgb.pkl


### Evaluate on Test Data

In [21]:
from helpers.evaluation import evaluate_on_test_data

df_test_preds, df_test_summary = evaluate_on_test_data(
    booster=booster, ntree=best_ntree,
    X_test=X_test, y_test=y_test,
    threshold=BUY_THRESHOLD, index_path=index_path,
    stocks_dir=stocks_dir, save_csv=str(RUN_OUTPUT_DIR / "test_predictions_full.csv"),
)

Saved test predictions -> /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/run_output/20260414_162450/test_predictions_full.csv
P(success | BUY): 28.76%  |  acc: 63.17%


# Optuna Hyperparameter Search (XGBoost)

In [29]:
import os, numpy as np, torch, optuna, wandb

optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ.update({"WANDB_SILENT": "true", "WANDB_CONSOLE": "off"})
os.makedirs(RUN_OUTPUT_DIR / "wandb", exist_ok=True)
wandb_settings = wandb.Settings(silent=True, quiet=True, console="off", root_dir=str(RUN_OUTPUT_DIR / "wandb"))
wandb.login(key="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")

# ── Config ───────────────────────────────────────────────────────────────────
OPTUNA_N_TRIALS   = 10
OPTUNA_EARLY_STOP = max(10, PATIENCE * 5)
OPTUNA_MAX_ROUNDS = 3000
WANDB_PROJECT, WANDB_GROUP = "NN-Trading-Bot", f"optuna_xgb_{SEED}"

dtrain_opt = xgb.DMatrix(X_train, label=y_train)
dval_opt   = xgb.DMatrix(X_val,   label=y_val)

TUNED_KEYS = ("max_depth", "eta", "subsample", "colsample_bytree", "min_child_weight", "gamma", "alpha", "lambda")
_fmt_val   = lambda v: f"{v:.4g}" if isinstance(v, float) else str(v)
_run_name  = lambda d: "____".join(f"{k}_{_fmt_val(d[k])}" for k in TUNED_KEYS)

def objective(trial):
    hp = {
        "max_depth":        trial.suggest_int("max_depth", 2, 8),
        "eta":              trial.suggest_float("eta", 1e-3, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "alpha":            trial.suggest_float("alpha", 0.0, 10.0),
        "lambda":           trial.suggest_float("lambda", 0.5, 10.0),
    }
    params = {"objective": "binary:logistic", "eval_metric": "logloss", "seed": SEED, "scale_pos_weight": scale_pos_weight, **GPU_XGB_PARAMS, **hp}

    try:
        run = wandb.init(project=WANDB_PROJECT, group=WANDB_GROUP, name=f"trial_{trial.number}__{_run_name(hp)}", config=hp, reinit=True, settings=wandb_settings)
    except Exception:
        run = None

    evals_res = {}
    bst = xgb.train(params=params, dtrain=dtrain_opt, num_boost_round=OPTUNA_MAX_ROUNDS,
                    evals=[(dtrain_opt, "train"), (dval_opt, "eval")],
                    early_stopping_rounds=OPTUNA_EARLY_STOP, evals_result=evals_res, verbose_eval=False)

    best_iter     = int(bst.best_iteration + 1) if bst.best_iteration is not None else OPTUNA_MAX_ROUNDS
    val_logloss   = evals_res["eval"]["logloss"][best_iter - 1]
    train_logloss = evals_res["train"]["logloss"][best_iter - 1]
    trial.set_user_attr("best_ntree", best_iter)
    trial.set_user_attr("val_logloss", val_logloss)

    if run is not None:
        for r, (tr, ev) in enumerate(zip(evals_res["train"]["logloss"], evals_res["eval"]["logloss"])):
            if (r + 1) % 100 == 0:
                wandb.log({"train/logloss": tr, "eval/logloss": ev, "round": r + 1})
        wandb.summary.update({"eval/best_ntree": best_iter, "train/final_logloss": train_logloss, "eval/final_logloss": val_logloss})
        run.finish()

    best_so_far = min((t.value for t in trial.study.trials if t.value is not None), default=val_logloss)
    marker = " *" if val_logloss <= best_so_far else ""
    print(f"  [{trial.number + 1:3d}/{OPTUNA_N_TRIALS}]  train={train_logloss:.6f}  val={val_logloss:.6f}  rounds={best_iter}{marker}")
    return val_logloss

study = optuna.create_study(direction="minimize", study_name="xgb_hparam_search", sampler=optuna.samplers.TPESampler(seed=SEED))
print(f"Starting Optuna search: {OPTUNA_N_TRIALS} trials …")
print(f"{'':>6}{'trial':>8}  {'train_loss':>12}  {'val_loss':>12}  {'rounds':>8}")
print(f"{'':>6}{'-'*8}  {'-'*12}  {'-'*12}  {'-'*8}")
study.optimize(objective, n_trials=OPTUNA_N_TRIALS, show_progress_bar=False)

best_trial  = study.best_trial
best_params = best_trial.params
print(f"\nBest trial #{best_trial.number}  val_logloss={best_trial.value:.6f}")
print("Best params:", best_params)

# ── Final run: retrain on train+val with best params ─────────────────────────
final_run = wandb.init(project=WANDB_PROJECT, group=WANDB_GROUP, name=f"BEST_trial_{best_trial.number}__{_run_name(best_params)}", config=best_params, reinit="finish_previous", settings=wandb_settings)

final_params = {"objective": "binary:logistic", "eval_metric": "logloss", "seed": SEED, "scale_pos_weight": scale_pos_weight, **GPU_XGB_PARAMS, **best_params}
best_ntree   = int(best_trial.user_attrs["best_ntree"])
dtrain_full  = xgb.DMatrix(np.concatenate([X_train, X_val]), label=np.concatenate([y_train, y_val]))

print(f"\nRetraining on train+val for {best_ntree} rounds …")
booster = xgb.train(params=final_params, dtrain=dtrain_full, num_boost_round=best_ntree, verbose_eval=False)

probs_test  = predict_probs_booster(booster, X_test,  best_ntree)
probs_train = predict_probs_booster(booster, X_train, best_ntree)
te_opt = buy_metrics(y_test,  probs_test,  BUY_THRESHOLD)
tr_opt = buy_metrics(y_train, probs_train, BUY_THRESHOLD)

print(f"Train  acc={tr_opt['acc']:.2f}%  P(success|BUY)={tr_opt['buy_success']:.2f}%")
print(f"Test   acc={te_opt['acc']:.2f}%  P(success|BUY)={te_opt['buy_success']:.2f}%  logloss={log_loss(y_test, probs_test):.6f}")
wandb.log({"train/final_logloss": log_loss(y_train, probs_train), "train/accuracy": tr_opt["acc"], "train/buy_success": tr_opt["buy_success"],
           "eval/test_logloss": log_loss(y_test, probs_test), "eval/test_accuracy": te_opt["acc"], "eval/test_buy_success": te_opt["buy_success"]})

MODEL_PATH = RUN_OUTPUT_DIR / "best_model_xgb.pkl"
joblib.dump({"booster": booster, "best_ntree": best_ntree}, MODEL_PATH)
print(f"\nSaved → {MODEL_PATH}")
print(f"Use BUY_THRESHOLD = {BUY_THRESHOLD}")
final_run.finish()
print("W&B runs finished.")


Starting Optuna search: 10 trials …
         trial    train_loss      val_loss    rounds
      --------  ------------  ------------  --------
  [  1/10]  train=0.515971  val=0.676542  rounds=37 *
  [  2/10]  train=0.460869  val=0.671645  rounds=105 *
  [  3/10]  train=0.525160  val=0.672766  rounds=388
  [  4/10]  train=0.535848  val=0.675980  rounds=49
  [  5/10]  train=0.597074  val=0.688882  rounds=82
  [  6/10]  train=0.675095  val=0.689045  rounds=101


[W 2026-04-14 16:33:21,997] Trial 6 failed with parameters: {'max_depth': 5, 'eta': 0.002870165242185818, 'subsample': 0.9847923138822793, 'colsample_bytree': 0.8650796940166687, 'min_child_weight': 19, 'gamma': 4.474136752138244, 'alpha': 5.978999788110851, 'lambda': 9.25780523271961} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/scratch/temp_/venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_405642/1380978226.py", line 41, in objective
    bst = xgb.train(params=params, dtrain=dtrain_opt, num_boost_round=OPTUNA_MAX_ROUNDS,
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/scratch/temp_/venv/lib/python3.12/site-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "/scratch/temp_/venv/lib/python3.12/site-packag

KeyboardInterrupt: 

# Eval Data Analysis

In [ ]:
from helpers.evaluation import evaluate_split_from_artifacts

SELECTED_THRESHOLD = 0.9

val_preds, daily, val_summary = evaluate_split_from_artifacts(
    split="val", threshold=SELECTED_THRESHOLD,
    index_path=index_path, scaler_path=scaler_path,
    model_path=str(RUN_OUTPUT_DIR / "best_model_xgb.pkl"), verbose=True,
)
display(daily[["Date", "num_trades", "pct_success", "pct_fail", "num_success", "num_fail"]])

if wandb.run is not None:
    wandb.log({
        "val_analysis/threshold":    SELECTED_THRESHOLD,
        "val_analysis/total_trades": int(val_summary["total_trades"]),
        "val_analysis/pct_success":  float(val_summary["pct_success"]),
        "val_analysis/daily_table":  wandb.Table(dataframe=daily.reset_index(drop=True)),
    })

Threshold : 0.900
Val days: 29  |  Total BUY trades: 1
P(success | BUY): 100.00%


,Date,num_trades,pct_success,pct_fail,num_success,num_fail
0,2025-12-17,1,100.0,0.0,1,0


# Test Data Analysis

In [ ]:
SELECTED_THRESHOLD = 0.9

test_preds, daily, test_summary = evaluate_split_from_artifacts(
    split="test", threshold=SELECTED_THRESHOLD,
    index_path=index_path, scaler_path=scaler_path,
    model_path=str(RUN_OUTPUT_DIR / "best_model_xgb.pkl"), verbose=True,
)
display(daily[["Date", "num_trades", "pct_success", "pct_fail", "num_success", "num_fail"]])

if wandb.run is not None:
    wandb.log({
        "test_analysis/threshold":    SELECTED_THRESHOLD,
        "test_analysis/total_trades": int(test_summary["total_trades"]),
        "test_analysis/pct_success":  float(test_summary["pct_success"]),
        "test_analysis/daily_table":  wandb.Table(dataframe=daily.reset_index(drop=True)),
    })

Threshold : 0.900
Test days: 52  |  Total BUY trades: 0
P(success | BUY): 0.00%


,Date,num_trades,pct_success,pct_fail,num_success,num_fail
